# ETL - Bronze Layer

Esta camada representa a ingestão de dados brutos (raw data) na arquitetura medalhão. Aqui, os dados são carregados do fonte e armazenados em formato Delta sem transformações significativas.

## Objetivos:
- Ingerir dados de fontes externas (ex.: CSV).
- Armazenar dados brutos em tabelas Delta.
- Manter a integridade dos dados originais.

In [ ]:
# Importar bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Criar sessão Spark (se não existir)
spark = SparkSession.builder.appName("Bronze Layer ETL").getOrCreate()

print("Sessão Spark criada com sucesso!")

In [ ]:
# Ingestão de dados brutos do CSV
bronze_df = spark.read.csv("../data/sample_data.csv", header=True, inferSchema=True)

# Adicionar metadados (ex.: timestamp de ingestão)
bronze_df = bronze_df.withColumn("ingestion_timestamp", current_timestamp())

# Mostrar amostra dos dados
bronze_df.show(5)
print(f"Total de registros: {bronze_df.count()}")

In [ ]:
# Salvar dados na camada Bronze (Delta Lake)
bronze_path = "/tmp/bronze/sample_data"
bronze_df.write.format("delta").mode("overwrite").save(bronze_path)

# Criar tabela Delta para facilitar consultas futuras
spark.sql(f"""
CREATE TABLE IF NOT EXISTS bronze_sample_data
USING DELTA
LOCATION '{bronze_path}'
""")

print("Dados salvos na camada Bronze com sucesso!")

In [ ]:
# Verificar tabela criada
spark.sql("DESCRIBE TABLE bronze_sample_data").show()
spark.sql("SELECT * FROM bronze_sample_data LIMIT 5").show()